In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

## Please use correct dataset path in the DATASET_PATH variable.

In [ ]:
DATASET_PATH = Path("../data/iavd_ml_dataset.parquet")

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"No dataset found at: {DATASET_PATH.resolve()}")

df = pd.read_parquet(DATASET_PATH)

In [ ]:
df.head()

## Please use correct mini-dataset path in the SOC_DATASET_PATH variable.

In [ ]:
SOC_DATASET_PATH = Path("../data/new_incidents_soc_1000.parquet")

In [ ]:
if not SOC_DATASET_PATH.exists():
    raise FileNotFoundError(f"No dataset found at: {SOC_DATASET_PATH.resolve()}")

# Загружаем маленький датасет
df_soc = pd.read_parquet(SOC_DATASET_PATH)
# Исключаем его записи из большого датасета
df = df[~df["EntityId"].isin(df_soc["EntityId"])].copy()

In [ ]:
# эти поля заполняются аналитиками ИБ в ходе расследования/ЖЦ
# необходимо их исключить из признаков, тк они дублируют таргет

columns_c = [
    "Дата закрытия (факт)",
    "Дата закрытия",
    "Дата взятия в работу",
    "Время Решения",
    "Время решения",
    "Период взятия в работу",
    "Ответсвенный по РП",
    "Группа устранения",
    "Фаза реагирования",
    "Статус инцидента из SIEM",
    "Ответственный",
    "Прошлый ответственный",
    "Полное имя",
    "Принятые меры",
    "Рекомендации по плейбуку",
    "Комментарий",
    "Решение",
    "Ошибки",
    "Ссылка в NAD (actual)",
    "Спам",
    "Эскалировался на L2?",
    "Пауза",
    "На мониторинг",
    "Заблокировать",
    "SLA просрочен (взятие в работу)",
    "Проверка на FP выполнена",
    "Выполнено первичное сдерживание",
    "Обогащение выполнено",
    "Классификация выполнена",
    "Успех brute force",
    "False Positive",
    "Легитимность действия",
    "Открыт для редактирования",
    "Белый список",
    "Ложное срабатывание",
    "Инцидент был изменен",
]

In [ ]:
existing_c = [col for col in columns_c if col in df.columns]
df_new = df.drop(columns=existing_c)
df_new.head()

In [ ]:
df_new = df_new.drop_duplicates()

constant_cols = [col for col in df_new.columns if df_new[col].nunique(dropna=True) <= 1]
df_new = df_new.drop(columns=constant_cols)

In [ ]:
df_new.head()

In [ ]:
df_new["Вердикт"].value_counts(dropna=False)

In [ ]:
# предлагается решить задачу бинарной классификации, поэтому нужно оставить
# только Решён и False Positive
df_new = df_new[df_new["Вердикт"].isin(["Решён", "False Positive"])].copy()

In [ ]:
df_new["y"] = df_new["Вердикт"].map({"Решён": 1, "False Positive": 0})

In [ ]:
df_new.dtypes

In [ ]:
df_new.dtypes.to_string()

In [ ]:
date_cols = [
    "Создан",
    "Время первого события",
    "Время последнего события",
    "Дата создания в источнике",
    "Дата изменения в источнике",
]

for col in date_cols:
    if col in df_new.columns:
        df_new[col] = pd.to_datetime(df_new[col], errors="coerce", utc=True)

In [ ]:
numeric_cols = [
    "Количество инцидентов",
    "SLA по устранению Атаки",
    "SLA по взятию в работу Атаки",
    "задержка перед автоназначением",
]

for col in numeric_cols:
    if col in df_new.columns:
        df_new[col] = pd.to_numeric(df_new[col], errors="coerce")

In [ ]:
df_new["hour"] = df_new["Создан"].dt.hour
df_new["day_of_week"] = df_new["Создан"].dt.dayofweek
df_new["month"] = df_new["Создан"].dt.month

In [ ]:
# продолжительность события:
df_new["event_duration_hours"] = (
    df_new["Время последнего события"] - df_new["Время первого события"]
).dt.total_seconds() / 3600

# упоминаются выбросы в датах, которые желательно убрать
threshold = pd.Timestamp("2024-04-12", tz="UTC")

invalid = (df_new["Время первого события"] < threshold) | (
    df_new["event_duration_hours"] < 0
)
df_new.loc[invalid, "event_duration_hours"] = float("nan")

In [ ]:
# время между созданием события в источнике и регистрацией карточки в SOAR
df_new["siem_soar_delay_hours"] = (
    df_new["Создан"] - df_new["Дата создания в источнике"]
).dt.total_seconds() / 3600

In [ ]:
numeric_features = [
    "Количество инцидентов",
    "SLA по устранению Атаки",
    "SLA по взятию в работу Атаки",
    "задержка перед автоназначением",
    "hour",
    "day_of_week",
    "month",
    "event_duration_hours",
    "siem_soar_delay_hours",
]

categorical_features = [
    "Критичность",
    "Источник инцидента",
    "Наименование",
    "Пиктограмма IRP",
]

numeric_features = [col for col in numeric_features if col in df_new.columns]

categorical_features = [col for col in categorical_features if col in df_new.columns]

In [ ]:
features = numeric_features + categorical_features
X = df_new[features]
y = df_new["y"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [ ]:
for X_part in [X_train, X_val, X_test]:
    for col in numeric_features:
        X_part[col] = pd.to_numeric(X_part[col], errors="coerce")
        X_part[col] = X_part[col].replace([np.inf, -np.inf], np.nan)

    for col in categorical_features:
        X_part[col] = X_part[col].fillna("Missing").astype(str)

In [ ]:
%pip install catboost

In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
)

In [ ]:
val_proba = model.predict_proba(X_val)[:, 1]
results = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred = (val_proba >= threshold).astype(int)

    results.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(y_val, y_pred, zero_division=0),
            "Recall": recall_score(y_val, y_pred, zero_division=0),
            "F1": f1_score(y_val, y_pred, zero_division=0),
            "Predicted positive": y_pred.sum(),
        }
    )

results_df = pd.DataFrame(results)
display(results_df.round(3))

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    results_df["Threshold"], results_df["Precision"], marker="o", label="Precision"
)
plt.plot(results_df["Threshold"], results_df["Recall"], marker="o", label="Recall")

plt.plot(results_df["Threshold"], results_df["F1"], marker="o", label="F1")

plt.xlabel("Порог классификации")
plt.ylabel("Значение метрики")
plt.title("Зависимость качества CatBoost от порога")
plt.xticks(results_df["Threshold"])
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# ВАЖНО: копия src/soc_dashboard/domain/ml_model_metadata.py, по ней дашборд
# читает metadata.json. Изменили поля здесь - поменяйте и там, и наоборот.
# Совпадение проверяет tests/domain/test_ml_model_metadata.py.
from dataclasses import dataclass


@dataclass(frozen=True)
class MLModelTrainingData:
    """Датасет, на котором обучалась модель."""

    file: str
    rows: int


@dataclass(frozen=True)
class MLModelMetrics:
    """Метрики на тестовой выборке."""

    accuracy: float
    precision: float
    recall: float
    f1: float


@dataclass(frozen=True)
class MLModelMetadata:
    """Содержимое ml_models/verdict/metadata.json."""

    # Что предсказывает модель
    task: str
    algorithm: str
    # Время обучения в UTC, ISO 8601
    trained_at: str
    data: MLModelTrainingData
    features: list[str]
    # Вероятность, начиная с которой прогноз - положительный класс
    threshold: float
    metrics: MLModelMetrics

    @classmethod
    def from_dict(cls, raw: dict) -> "MLModelMetadata":
        """Читает словарь из metadata.json."""
        return cls(
            **{
                **raw,
                "data": MLModelTrainingData(**raw["data"]),
                "metrics": MLModelMetrics(**raw["metrics"]),
            }
        )

In [ ]:
import json
from dataclasses import asdict

from sklearn.metrics import accuracy_score

# Порог выбран по валидации (таблица и график выше), метрики считаются на тесте
THRESHOLD = 0.7

test_pred = (model.predict_proba(X_test)[:, 1] >= THRESHOLD).astype(int)

metadata = MLModelMetadata(
    task="Решён или False Positive",
    algorithm="CatBoostClassifier",
    trained_at=pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    data=MLModelTrainingData(file=DATASET_PATH.name, rows=len(df_new)),
    features=features,
    threshold=THRESHOLD,
    metrics=MLModelMetrics(
        accuracy=round(float(accuracy_score(y_test, test_pred)), 4),
        precision=round(float(precision_score(y_test, test_pred)), 4),
        recall=round(float(recall_score(y_test, test_pred)), 4),
        f1=round(float(f1_score(y_test, test_pred)), 4),
    ),
)

OUT = Path("../ml_models/verdict")
OUT.mkdir(parents=True, exist_ok=True)

joblib.dump(model, OUT / "model.joblib")
(OUT / "metadata.json").write_text(
    json.dumps(asdict(metadata), ensure_ascii=False, indent=2), encoding="utf-8"
)